In [13]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET

from datetime import datetime, timezone
from getpass import getpass
from uuid import uuid4
from pathlib import Path

TOKEN = getpass("OJP API Token: ").strip()

if not TOKEN:
    raise ValueError("Der OJP API Token ist leer.")

URL = "https://api.opentransportdata.swiss/ojp20"

STOP_ID = "ch:1:sloid:3000"
STOP_NAME = "Zürich HB"

now_utc = datetime.now(timezone.utc)
timestamp = now_utc.isoformat(timespec="milliseconds").replace("+00:00", "Z")

message_id = f"zhaw-pilot-{uuid4()}"

xml_request = f"""<?xml version="1.0" encoding="UTF-8"?>
<OJP
    xmlns="http://www.vdv.de/ojp"
    xmlns:siri="http://www.siri.org.uk/siri"
    xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
    xmlns:xsd="http://www.w3.org/2001/XMLSchema"
    xsi:schemaLocation="http://www.vdv.de/ojp"
    version="2.0">

    <OJPRequest>
        <siri:ServiceRequest>

            <siri:ServiceRequestContext>
                <siri:Language>de</siri:Language>
            </siri:ServiceRequestContext>

            <siri:RequestTimestamp>{timestamp}</siri:RequestTimestamp>

            <siri:RequestorRef>ZHAW_DataAnalytics_Project</siri:RequestorRef>

            <OJPStopEventRequest>

                <siri:RequestTimestamp>{timestamp}</siri:RequestTimestamp>

                <siri:MessageIdentifier>{message_id}</siri:MessageIdentifier>

                <Location>

                    <PlaceRef>

                        <siri:StopPointRef>{STOP_ID}</siri:StopPointRef>

                        <Name>
                            <Text>{STOP_NAME}</Text>
                        </Name>

                    </PlaceRef>

                    <DepArrTime>{timestamp}</DepArrTime>

                </Location>

                <Params>

                    <NumberOfResults>30</NumberOfResults>

                    <StopEventType>departure</StopEventType>

                    <IncludePreviousCalls>false</IncludePreviousCalls>

                    <IncludeOnwardCalls>false</IncludeOnwardCalls>

                    <UseRealtimeData>full</UseRealtimeData>

                </Params>

            </OJPStopEventRequest>

        </siri:ServiceRequest>
    </OJPRequest>

</OJP>
"""

headers = {
    "Content-Type": "application/xml",
    "Authorization": f"Bearer {TOKEN}"
}

response = requests.post(
    URL,
    headers=headers,
    data=xml_request.encode("utf-8"),
    timeout=30
)



In [4]:
import xml.etree.ElementTree as ET

# XML-Antwort in eine durchsuchbare Struktur umwandeln
root = ET.fromstring(response.content)

# XML namespaces
ns = {
    "ojp": "http://www.vdv.de/ojp",
    "siri": "http://www.siri.org.uk/siri"
}

# Alle gefundenen Verbindungen / Stop Events suchen
results = root.findall(".//ojp:StopEventResult", ns)

print("Gefundene Verbindungen:", len(results))



Gefundene Verbindungen: 30


In [5]:
import pandas as pd
import xml.etree.ElementTree as ET

# XML erneut parsen
root = ET.fromstring(response.content)

# Namespaces
ns = {
    "ojp": "http://www.vdv.de/ojp",
    "siri": "http://www.siri.org.uk/siri"
}

# Alle StopEventResult-Elemente finden
results = root.findall(".//ojp:StopEventResult", ns)

rows = []

for result in results:

    stop_event = result.find("ojp:StopEvent", ns)

    if stop_event is None:
        continue

    # --------------------------------------------------
    # 1. Informationen zum aktuellen Halt
    # --------------------------------------------------

    this_call = stop_event.find(
        "ojp:ThisCall/ojp:CallAtStop",
        ns
    )

    if this_call is None:
        continue

    stop_point_ref = this_call.findtext(
        "siri:StopPointRef",
        default=None,
        namespaces=ns
    )

    stop_name = this_call.findtext(
        "ojp:StopPointName/ojp:Text",
        default=None,
        namespaces=ns
    )

    planned_platform = this_call.findtext(
        "ojp:PlannedQuay/ojp:Text",
        default=None,
        namespaces=ns
    )

    scheduled_departure = this_call.findtext(
        "ojp:ServiceDeparture/ojp:TimetabledTime",
        default=None,
        namespaces=ns
    )

    estimated_departure = this_call.findtext(
        "ojp:ServiceDeparture/ojp:EstimatedTime",
        default=None,
        namespaces=ns
    )

    # --------------------------------------------------
    # 2. Informationen zur Fahrt
    # --------------------------------------------------

    service = stop_event.find(
        "ojp:Service",
        ns
    )

    if service is None:
        continue

    operating_day = service.findtext(
        "ojp:OperatingDayRef",
        default=None,
        namespaces=ns
    )

    journey_ref = service.findtext(
        "ojp:JourneyRef",
        default=None,
        namespaces=ns
    )

    public_code = service.findtext(
        "ojp:PublicCode",
        default=None,
        namespaces=ns
    )

    transport_mode = service.findtext(
        "ojp:Mode/ojp:PtMode",
        default=None,
        namespaces=ns
    )

    product_category = service.findtext(
        "ojp:ProductCategory/ojp:Name/ojp:Text",
        default=None,
        namespaces=ns
    )

    line = service.findtext(
        "ojp:PublishedServiceName/ojp:Text",
        default=None,
        namespaces=ns
    )

    train_number = service.findtext(
        "ojp:TrainNumber",
        default=None,
        namespaces=ns
    )

    origin = service.findtext(
        "ojp:OriginText/ojp:Text",
        default=None,
        namespaces=ns
    )

    destination = service.findtext(
        "ojp:DestinationText/ojp:Text",
        default=None,
        namespaces=ns
    )

    # --------------------------------------------------
    # 3. Als Zeile speichern
    # --------------------------------------------------

    rows.append({
        "operating_day": operating_day,
        "journey_ref": journey_ref,
        "stop_point_ref": stop_point_ref,
        "station_name": stop_name,
        "transport_mode": transport_mode,
        "product_category": product_category,
        "public_code": public_code,
        "line": line,
        "train_number": train_number,
        "origin": origin,
        "destination": destination,
        "planned_platform": planned_platform,
        "scheduled_departure": scheduled_departure,
        "estimated_departure": estimated_departure
    })


# ============================================================
# DATAFRAME ERSTELLEN
# ============================================================

df = pd.DataFrame(rows)

print("Anzahl Verbindungen:", len(df))

display(df)

# Zeitspalten in echte datetime-Werte umwandeln

df["scheduled_departure"] = pd.to_datetime(
    df["scheduled_departure"],
    utc=True,
    errors="coerce"
)

df["estimated_departure"] = pd.to_datetime(
    df["estimated_departure"],
    utc=True,
    errors="coerce"
)


# Schweizer Lokalzeit erzeugen

df["scheduled_departure_local"] = (
    df["scheduled_departure"]
    .dt.tz_convert("Europe/Zurich")
)

df["estimated_departure_local"] = (
    df["estimated_departure"]
    .dt.tz_convert("Europe/Zurich")
)


# Prüfen, ob Echtzeitinformation vorhanden ist

df["has_realtime"] = (
    df["estimated_departure"]
    .notna()
)


# Verspätung berechnen

df["predicted_delay_minutes"] = (
    df["estimated_departure"]
    - df["scheduled_departure"]
).dt.total_seconds() / 60


# Wichtige Spalten anzeigen

display(
    df[
        [
            "station_name",
            "transport_mode",
            "product_category",
            "line",
            "origin",
            "destination",
            "planned_platform",
            "scheduled_departure_local",
            "estimated_departure_local",
            "predicted_delay_minutes"
        ]
    ]
)



Anzahl Verbindungen: 30


,operating_day,journey_ref,stop_point_ref,station_name,transport_mode,product_category,public_code,line,train_number,origin,destination,planned_platform,scheduled_departure,estimated_departure
0,2026-09-21,ch:1:sjyid:100001:18263-001,ch:1:sloid:3000:500:31,Zürich HB,rail,S-Bahn,S2,S2,18263,Zürich Flughafen,Ziegelbrücke,32,2026-09-21T14:47:00Z,2026-09-21T14:48:00Z
1,2026-09-21,ch:1:sjyid:100001:19462-001,ch:1:sloid:3000:500:32,Zürich HB,rail,S-Bahn,S14,S14,19462,Hinwil,Affoltern am Albis,31,2026-09-21T14:49:00Z,2026-09-21T14:50:00Z
2,2026-09-21,ch:1:sjyid:100001:19963-001,ch:1:sloid:3000:501:34,Zürich HB,rail,S-Bahn,S19,S19,19963,Koblenz,Pfäffikon ZH,34,2026-09-21T14:49:00Z,2026-09-21T14:49:18Z
3,2026-09-21,ch:1:sjyid:100001:18762-001,ch:1:sloid:3000:502:42,Zürich HB,rail,S-Bahn,S7,S7,18762,Rapperswil SG,Winterthur,41/42,2026-09-21T14:49:00Z,2026-09-21T14:49:42Z
4,2026-09-21,ch:1:sjyid:100001:20463-001,ch:1:sloid:3000:3:4,Zürich HB,rail,S-Bahn,S24,S24,20463,Weinfelden,Zug,4,2026-09-21T14:51:00Z,2026-09-21T14:51:18Z
5,2026-09-21,ch:1:sjyid:100001:2077-001,ch:1:sloid:3000:501:33,Zürich HB,rail,InterRegio,IR36,IR36,2077,Basel SBB,Zürich Flughafen,33,2026-09-21T14:52:00Z,2026-09-21T14:52:24Z
6,2026-09-21,ch:1:sjyid:100001:19562-001,ch:1:sloid:3000:502:42,Zürich HB,rail,S-Bahn,S15,S15,19562,Rapperswil SG,Niederweningen,41/42,2026-09-21T14:52:00Z,2026-09-21T14:53:18Z
7,2026-09-21,ch:1:sjyid:100061:2376-001,ch:1:sloid:3000:8:15,Zürich HB,rail,Aare Linth,IR35,IR35,2376,Chur,Bern,15,2026-09-21T14:53:00Z,2026-09-21T14:53:30Z
8,2026-09-21,ch:1:sjyid:100001:18563-001,ch:1:sloid:3000:503:43,Zürich HB,rail,S-Bahn,S5,S5,18563,Zug,Pfäffikon SZ,43/44,2026-09-21T14:54:00Z,2026-09-21T14:57:00Z
9,2026-09-21,ch:1:sjyid:100001:18862-001,ch:1:sloid:3000:501:34,Zürich HB,rail,S-Bahn,S8,S8,18862,Pfäffikon SZ,Winterthur,34,2026-09-21T14:55:00Z,2026-09-21T14:55:18Z


,station_name,transport_mode,product_category,line,origin,destination,planned_platform,scheduled_departure_local,estimated_departure_local,predicted_delay_minutes
0,Zürich HB,rail,S-Bahn,S2,Zürich Flughafen,Ziegelbrücke,32,2026-09-21 16:47:00+02:00,2026-09-21 16:48:00+02:00,1.0
1,Zürich HB,rail,S-Bahn,S14,Hinwil,Affoltern am Albis,31,2026-09-21 16:49:00+02:00,2026-09-21 16:50:00+02:00,1.0
2,Zürich HB,rail,S-Bahn,S19,Koblenz,Pfäffikon ZH,34,2026-09-21 16:49:00+02:00,2026-09-21 16:49:18+02:00,0.3
3,Zürich HB,rail,S-Bahn,S7,Rapperswil SG,Winterthur,41/42,2026-09-21 16:49:00+02:00,2026-09-21 16:49:42+02:00,0.7
4,Zürich HB,rail,S-Bahn,S24,Weinfelden,Zug,4,2026-09-21 16:51:00+02:00,2026-09-21 16:51:18+02:00,0.3
5,Zürich HB,rail,InterRegio,IR36,Basel SBB,Zürich Flughafen,33,2026-09-21 16:52:00+02:00,2026-09-21 16:52:24+02:00,0.4
6,Zürich HB,rail,S-Bahn,S15,Rapperswil SG,Niederweningen,41/42,2026-09-21 16:52:00+02:00,2026-09-21 16:53:18+02:00,1.3
7,Zürich HB,rail,Aare Linth,IR35,Chur,Bern,15,2026-09-21 16:53:00+02:00,2026-09-21 16:53:30+02:00,0.5
8,Zürich HB,rail,S-Bahn,S5,Zug,Pfäffikon SZ,43/44,2026-09-21 16:54:00+02:00,2026-09-21 16:57:00+02:00,3.0
9,Zürich HB,rail,S-Bahn,S8,Pfäffikon SZ,Winterthur,34,2026-09-21 16:55:00+02:00,2026-09-21 16:55:18+02:00,0.3


In [6]:
from pathlib import Path
import pandas as pd

# Zeitpunkt dieses API-Abrufs speichern
collection_time = pd.Timestamp.now(tz="UTC")

df["collection_timestamp"] = collection_time

df["collection_timestamp_local"] = (
    df["collection_timestamp"]
    .dt.tz_convert("Europe/Zurich")
)

# Ordner erstellen, falls er noch nicht existiert
snapshot_dir = Path("data/interim")
snapshot_dir.mkdir(parents=True, exist_ok=True)

# Eindeutiger Dateiname
snapshot_filename = (
    snapshot_dir
    / f"zuerich_hb_{collection_time.strftime('%Y%m%d_%H%M%S')}.csv"
)

# Snapshot speichern
df.to_csv(
    snapshot_filename,
    index=False
)

print("Snapshot gespeichert:")
print(snapshot_filename)

display(
    df[
        [
            "collection_timestamp_local",
            "journey_ref",
            "line",
            "destination",
            "scheduled_departure_local",
            "estimated_departure_local",
            "predicted_delay_minutes"
        ]
    ]
)

Snapshot gespeichert:
data/interim/zuerich_hb_20260921_144900.csv


,collection_timestamp_local,journey_ref,line,destination,scheduled_departure_local,estimated_departure_local,predicted_delay_minutes
0,2026-09-21 16:49:00.737979+02:00,ch:1:sjyid:100001:18263-001,S2,Ziegelbrücke,2026-09-21 16:47:00+02:00,2026-09-21 16:48:00+02:00,1.0
1,2026-09-21 16:49:00.737979+02:00,ch:1:sjyid:100001:19462-001,S14,Affoltern am Albis,2026-09-21 16:49:00+02:00,2026-09-21 16:50:00+02:00,1.0
2,2026-09-21 16:49:00.737979+02:00,ch:1:sjyid:100001:19963-001,S19,Pfäffikon ZH,2026-09-21 16:49:00+02:00,2026-09-21 16:49:18+02:00,0.3
3,2026-09-21 16:49:00.737979+02:00,ch:1:sjyid:100001:18762-001,S7,Winterthur,2026-09-21 16:49:00+02:00,2026-09-21 16:49:42+02:00,0.7
4,2026-09-21 16:49:00.737979+02:00,ch:1:sjyid:100001:20463-001,S24,Zug,2026-09-21 16:51:00+02:00,2026-09-21 16:51:18+02:00,0.3
5,2026-09-21 16:49:00.737979+02:00,ch:1:sjyid:100001:2077-001,IR36,Zürich Flughafen,2026-09-21 16:52:00+02:00,2026-09-21 16:52:24+02:00,0.4
6,2026-09-21 16:49:00.737979+02:00,ch:1:sjyid:100001:19562-001,S15,Niederweningen,2026-09-21 16:52:00+02:00,2026-09-21 16:53:18+02:00,1.3
7,2026-09-21 16:49:00.737979+02:00,ch:1:sjyid:100061:2376-001,IR35,Bern,2026-09-21 16:53:00+02:00,2026-09-21 16:53:30+02:00,0.5
8,2026-09-21 16:49:00.737979+02:00,ch:1:sjyid:100001:18563-001,S5,Pfäffikon SZ,2026-09-21 16:54:00+02:00,2026-09-21 16:57:00+02:00,3.0
9,2026-09-21 16:49:00.737979+02:00,ch:1:sjyid:100001:18862-001,S8,Winterthur,2026-09-21 16:55:00+02:00,2026-09-21 16:55:18+02:00,0.3


In [7]:
from pathlib import Path
import pandas as pd

snapshot_files = sorted(
    Path("data/interim").glob("zuerich_hb_*.csv")
)

print("Gefundene Snapshots:", len(snapshot_files))

if len(snapshot_files) < 2:
    print("Noch mindestens einen zweiten Snapshot erstellen.")

else:

    old_file = snapshot_files[-2]
    new_file = snapshot_files[-1]

    print("Alter Snapshot:", old_file.name)
    print("Neuer Snapshot:", new_file.name)

    old_df = pd.read_csv(old_file)
    new_df = pd.read_csv(new_file)

    # Dieselbe Fahrt anhand eindeutiger Merkmale erkennen
    keys = [
        "journey_ref",
        "operating_day",
        "station_name",
        "scheduled_departure"
    ]

    comparison = old_df.merge(
        new_df,
        on=keys,
        how="inner",
        suffixes=("_old", "_new")
    )

    comparison["delay_change_minutes"] = (
        comparison["predicted_delay_minutes_new"]
        - comparison["predicted_delay_minutes_old"]
    )

    print(
        "Fahrten, die in beiden Snapshots vorkommen:",
        len(comparison)
    )

    display(
        comparison[
            [
                "line_old",
                "destination_old",
                "scheduled_departure",
                "predicted_delay_minutes_old",
                "predicted_delay_minutes_new",
                "delay_change_minutes"
            ]
        ]
    )

Gefundene Snapshots: 4
Alter Snapshot: zuerich_hb_20260919_102350.csv
Neuer Snapshot: zuerich_hb_20260921_144900.csv
Fahrten, die in beiden Snapshots vorkommen: 0


,line_old,destination_old,scheduled_departure,predicted_delay_minutes_old,predicted_delay_minutes_new,delay_change_minutes


In [8]:
from pathlib import Path
import pandas as pd

snapshot_files = sorted(
    Path("data/interim").glob("zuerich_hb_*.csv")
)

old_df = pd.read_csv(snapshot_files[-2])
new_df = pd.read_csv(snapshot_files[-1])

print("ALTER SNAPSHOT")
print("Anzahl:", len(old_df))
print(
    "Abfahrten von:",
    old_df["scheduled_departure"].min(),
    "bis:",
    old_df["scheduled_departure"].max()
)

print("\nNEUER SNAPSHOT")
print("Anzahl:", len(new_df))
print(
    "Abfahrten von:",
    new_df["scheduled_departure"].min(),
    "bis:",
    new_df["scheduled_departure"].max()
)

old_journeys = set(old_df["journey_ref"].dropna())
new_journeys = set(new_df["journey_ref"].dropna())

common_journeys = old_journeys.intersection(new_journeys)

print("\nGemeinsame journey_ref:")
print(len(common_journeys))

print("\nJourneyRefs alter Snapshot:")
print(old_df[["line", "scheduled_departure", "journey_ref"]])

print("\nJourneyRefs neuer Snapshot:")
print(new_df[["line", "scheduled_departure", "journey_ref"]])

ALTER SNAPSHOT
Anzahl: 30
Abfahrten von: 2026-09-19 10:24:00+00:00 bis: 2026-09-19 10:46:00+00:00

NEUER SNAPSHOT
Anzahl: 30
Abfahrten von: 2026-09-21 14:47:00+00:00 bis: 2026-09-21 15:07:00+00:00

Gemeinsame journey_ref:
0

JourneyRefs alter Snapshot:
    line        scheduled_departure                  journey_ref
0     S5  2026-09-19 10:24:00+00:00  ch:1:sjyid:100001:18545-001
1     S8  2026-09-19 10:25:00+00:00  ch:1:sjyid:100001:18844-001
2     S9  2026-09-19 10:28:00+00:00  ch:1:sjyid:100001:18945-001
3   IR55  2026-09-19 10:29:00+00:00   ch:1:sjyid:100001:1770-001
4    S11  2026-09-19 10:29:00+00:00  ch:1:sjyid:100001:19146-001
5     S6  2026-09-19 10:30:00+00:00  ch:1:sjyid:100001:18645-001
6     S6  2026-09-19 10:31:00+00:00  ch:1:sjyid:100001:18646-001
7    IC1  2026-09-19 10:32:00+00:00    ch:1:sjyid:100001:718-001
8     EC  2026-09-19 10:33:00+00:00    ch:1:sjyid:100001:151-001
9    IC3  2026-09-19 10:34:00+00:00    ch:1:sjyid:100001:568-001
10    S3  2026-09-19 10:34:00+00

In [9]:
# ============================================================
# SNAPSHOT COMPARISON
# ============================================================

keys = [
    "operating_day",
    "journey_ref"
]

comparison = old_df.merge(
    new_df,
    on=keys,
    how="inner",
    suffixes=("_old", "_new")
)

comparison["delay_change_minutes"] = (
    comparison["predicted_delay_minutes_new"]
    - comparison["predicted_delay_minutes_old"]
)

print(
    "Fahrten in beiden Snapshots:",
    len(comparison)
)

display(
    comparison[
        [
            "line_old",
            "destination_old",
            "scheduled_departure_old",
            "predicted_delay_minutes_old",
            "predicted_delay_minutes_new",
            "delay_change_minutes"
        ]
    ]
)

Fahrten in beiden Snapshots: 0


,line_old,destination_old,scheduled_departure_old,predicted_delay_minutes_old,predicted_delay_minutes_new,delay_change_minutes


In [14]:
def collect_ojp_station(
    stop_id,
    stop_name,
    token,
    number_of_results=30
):
    """
    Ruft OJP-Echtzeitdaten für eine Haltestelle ab,
    parst die XML-Antwort und gibt einen pandas DataFrame zurück.
    """

    # --------------------------------------------------------
    # 1. Zeitpunkt des API-Abrufs
    # --------------------------------------------------------

    now_utc = datetime.now(timezone.utc)

    timestamp = (
        now_utc
        .isoformat(timespec="milliseconds")
        .replace("+00:00", "Z")
    )

    message_id = f"zhaw-project-{uuid4()}"

    # --------------------------------------------------------
    # 2. XML Request
    # --------------------------------------------------------

    xml_request = f"""<?xml version="1.0" encoding="UTF-8"?>
<OJP
    xmlns="http://www.vdv.de/ojp"
    xmlns:siri="http://www.siri.org.uk/siri"
    xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
    xmlns:xsd="http://www.w3.org/2001/XMLSchema"
    xsi:schemaLocation="http://www.vdv.de/ojp"
    version="2.0">

    <OJPRequest>
        <siri:ServiceRequest>

            <siri:ServiceRequestContext>
                <siri:Language>de</siri:Language>
            </siri:ServiceRequestContext>

            <siri:RequestTimestamp>{timestamp}</siri:RequestTimestamp>

            <siri:RequestorRef>ZHAW_DataAnalytics_Project</siri:RequestorRef>

            <OJPStopEventRequest>

                <siri:RequestTimestamp>{timestamp}</siri:RequestTimestamp>

                <siri:MessageIdentifier>{message_id}</siri:MessageIdentifier>

                <Location>

                    <PlaceRef>

                        <siri:StopPointRef>{stop_id}</siri:StopPointRef>

                        <Name>
                            <Text>{stop_name}</Text>
                        </Name>

                    </PlaceRef>

                    <DepArrTime>{timestamp}</DepArrTime>

                </Location>

                <Params>

                    <NumberOfResults>{number_of_results}</NumberOfResults>

                    <StopEventType>departure</StopEventType>

                    <IncludePreviousCalls>false</IncludePreviousCalls>

                    <IncludeOnwardCalls>false</IncludeOnwardCalls>

                    <UseRealtimeData>full</UseRealtimeData>

                </Params>

            </OJPStopEventRequest>

        </siri:ServiceRequest>
    </OJPRequest>

</OJP>
"""

    # --------------------------------------------------------
    # 3. API Request senden
    # --------------------------------------------------------

    headers = {
        "Content-Type": "application/xml",
        "Authorization": f"Bearer {token}"
    }

    response = requests.post(
        URL,
        headers=headers,
        data=xml_request.encode("utf-8"),
        timeout=30
    )

    print(
        f"{stop_name}: HTTP {response.status_code}"
    )

    # Falls die API einen Fehler liefert:
    if response.status_code != 200:
        print("\nAPI-Fehler:")
        print(response.text[:1000])

    response.raise_for_status()

    # --------------------------------------------------------
    # 4. Raw XML speichern
    # --------------------------------------------------------

    safe_station_name = (
        stop_name
        .lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("ü", "ue")
        .replace("ö", "oe")
        .replace("ä", "ae")
        .replace("é", "e")
        .replace("è", "e")
    )

    raw_dir = Path("data/raw/ojp")

    raw_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    raw_filename = (
        raw_dir
        / f"{safe_station_name}_"
          f"{now_utc.strftime('%Y%m%d_%H%M%S')}.xml"
    )

    raw_filename.write_bytes(
        response.content
    )

    # --------------------------------------------------------
    # 5. XML parsen
    # --------------------------------------------------------

    root = ET.fromstring(
        response.content
    )

    ns = {
        "ojp": "http://www.vdv.de/ojp",
        "siri": "http://www.siri.org.uk/siri"
    }

    results = root.findall(
        ".//ojp:StopEventResult",
        ns
    )

    print(
        f"{stop_name}: {len(results)} Verbindungen"
    )

    # --------------------------------------------------------
    # 6. Daten extrahieren
    # --------------------------------------------------------

    rows = []

    for result in results:

        stop_event = result.find(
            "ojp:StopEvent",
            ns
        )

        if stop_event is None:
            continue

        this_call = stop_event.find(
            "ojp:ThisCall/ojp:CallAtStop",
            ns
        )

        service = stop_event.find(
            "ojp:Service",
            ns
        )

        if this_call is None or service is None:
            continue

        rows.append({

            "collection_timestamp": timestamp,

            "station_id": stop_id,

            "station_name": stop_name,

            "stop_point_ref": this_call.findtext(
                "siri:StopPointRef",
                default=None,
                namespaces=ns
            ),

            "operating_day": service.findtext(
                "ojp:OperatingDayRef",
                default=None,
                namespaces=ns
            ),

            "journey_ref": service.findtext(
                "ojp:JourneyRef",
                default=None,
                namespaces=ns
            ),

            "transport_mode": service.findtext(
                "ojp:Mode/ojp:PtMode",
                default=None,
                namespaces=ns
            ),

            "product_category": service.findtext(
                "ojp:ProductCategory/ojp:Name/ojp:Text",
                default=None,
                namespaces=ns
            ),

            "public_code": service.findtext(
                "ojp:PublicCode",
                default=None,
                namespaces=ns
            ),

            "line": service.findtext(
                "ojp:PublishedServiceName/ojp:Text",
                default=None,
                namespaces=ns
            ),

            "train_number": service.findtext(
                "ojp:TrainNumber",
                default=None,
                namespaces=ns
            ),

            "origin": service.findtext(
                "ojp:OriginText/ojp:Text",
                default=None,
                namespaces=ns
            ),

            "destination": service.findtext(
                "ojp:DestinationText/ojp:Text",
                default=None,
                namespaces=ns
            ),

            "planned_platform": this_call.findtext(
                "ojp:PlannedQuay/ojp:Text",
                default=None,
                namespaces=ns
            ),

            "estimated_platform": this_call.findtext(
                "ojp:EstimatedQuay/ojp:Text",
                default=None,
                namespaces=ns
            ),

            "scheduled_departure": this_call.findtext(
                "ojp:ServiceDeparture/ojp:TimetabledTime",
                default=None,
                namespaces=ns
            ),

            "estimated_departure": this_call.findtext(
                "ojp:ServiceDeparture/ojp:EstimatedTime",
                default=None,
                namespaces=ns
            )
        })

    # --------------------------------------------------------
    # 7. DataFrame erstellen
    # --------------------------------------------------------

    df = pd.DataFrame(rows)

    if df.empty:
        print(
            f"Keine Daten für {stop_name} gefunden."
        )

        return df

    # --------------------------------------------------------
    # 8. Zeitvariablen umwandeln
    # --------------------------------------------------------

    time_columns = [
        "collection_timestamp",
        "scheduled_departure",
        "estimated_departure"
    ]

    for column in time_columns:

        df[column] = pd.to_datetime(
            df[column],
            utc=True,
            errors="coerce"
        )

    # --------------------------------------------------------
    # 9. Realtime und Delay berechnen
    # --------------------------------------------------------

    df["has_realtime"] = (
        df["estimated_departure"]
        .notna()
    )

    # Fehlende EstimatedTime bleibt NaN.
    # Sie wird NICHT als 0 Minuten Verspätung interpretiert.

    df["predicted_delay_minutes"] = (
        df["estimated_departure"]
        - df["scheduled_departure"]
    ).dt.total_seconds() / 60

    # --------------------------------------------------------
    # 10. Schweizer Lokalzeit
    # --------------------------------------------------------

    df["collection_timestamp_local"] = (
        df["collection_timestamp"]
        .dt.tz_convert("Europe/Zurich")
    )

    df["scheduled_departure_local"] = (
        df["scheduled_departure"]
        .dt.tz_convert("Europe/Zurich")
    )

    df["estimated_departure_local"] = (
        df["estimated_departure"]
        .dt.tz_convert("Europe/Zurich")
    )

    # --------------------------------------------------------
    # 11. Zeitliche Merkmale
    # --------------------------------------------------------

    df["date"] = (
        df["scheduled_departure_local"]
        .dt.date
    )

    df["hour"] = (
        df["scheduled_departure_local"]
        .dt.hour
    )

    df["weekday"] = (
        df["scheduled_departure_local"]
        .dt.day_name()
    )

    df["weekend"] = (
        df["scheduled_departure_local"]
        .dt.dayofweek >= 5
    )

    df["minutes_until_departure"] = (
        df["scheduled_departure"]
        - df["collection_timestamp"]
    ).dt.total_seconds() / 60

    # --------------------------------------------------------
    # 12. CSV Snapshot speichern
    # --------------------------------------------------------

    interim_dir = Path(
        "data/interim"
    )

    interim_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    csv_filename = (
        interim_dir
        / f"{safe_station_name}_"
          f"{now_utc.strftime('%Y%m%d_%H%M%S')}.csv"
    )

    df.to_csv(
        csv_filename,
        index=False
    )

    print(
        f"Gespeichert: {csv_filename}"
    )

    return df

In [15]:
df_zurich = collect_ojp_station(
    stop_id="ch:1:sloid:3000",
    stop_name="Zürich HB",
    token=TOKEN,
    number_of_results=30
)

Zürich HB: HTTP 200
Zürich HB: 30 Verbindungen
Gespeichert: data/interim/zuerich_hb_20260921_150719.csv


In [16]:
display(
    df_zurich[
        [
            "station_name",
            "transport_mode",
            "product_category",
            "line",
            "destination",
            "scheduled_departure_local",
            "estimated_departure_local",
            "predicted_delay_minutes"
        ]
    ].head(10)
)

,station_name,transport_mode,product_category,line,destination,scheduled_departure_local,estimated_departure_local,predicted_delay_minutes
0,Zürich HB,rail,RegioExpress,RE48,Schaffhausen,2026-09-21 17:05:00+02:00,2026-09-21 17:07:00+02:00,2.0
1,Zürich HB,rail,S-Bahn,S23,Winterthur,2026-09-21 17:06:00+02:00,2026-09-21 17:09:06+02:00,3.1
2,Zürich HB,rail,S-Bahn,S8,Pfäffikon SZ,2026-09-21 17:07:00+02:00,2026-09-21 17:08:00+02:00,1.0
3,Zürich HB,rail,S-Bahn,S9,Schaffhausen,2026-09-21 17:07:00+02:00,2026-09-21 17:07:18+02:00,0.3
4,Zürich HB,rail,InterCity,IC3,Chur,2026-09-21 17:07:00+02:00,2026-09-21 17:08:00+02:00,1.0
5,Zürich HB,rail,InterRegio,IR37,Basel SBB,2026-09-21 17:08:00+02:00,2026-09-21 17:08:30+02:00,0.5
6,Zürich HB,rail,InterRegio,IR13,Sargans,2026-09-21 17:08:00+02:00,2026-09-21 17:08:30+02:00,0.5
7,Zürich HB,rail,S-Bahn,S15,Rapperswil SG,2026-09-21 17:09:00+02:00,2026-09-21 17:10:12+02:00,1.2
8,Zürich HB,rail,S-Bahn,S5,Zug,2026-09-21 17:09:00+02:00,2026-09-21 17:09:18+02:00,0.3
9,Zürich HB,rail,InterRegio,IR70,Luzern,2026-09-21 17:10:00+02:00,2026-09-21 17:10:30+02:00,0.5
